##### Data Ingestion

In [ ]:
# .\payroll_venv\Scripts\activate inside cd 2_payroll-agentic-ai & deactivate command to come out
# python -m ipykernel install --user --name=payroll_venv --display-name "payroll_venv" kernel registered

In [1]:
import sys
print(sys.executable)

c:\Users\Lenovo\Downloads\agentic AI practice\3_root\backend\agentic_env\Scripts\python.exe


##### **Upload CSVs/docs to Azure Blob**

In [2]:
# Upload CSVs/docs to Azure Blob
from azure.storage.blob import BlobServiceClient
import os
from dotenv import load_dotenv

load_dotenv(override=True)

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

containers = ["raw-data", "documents", "processed-data"]

for container in containers:
    try:
        blob_service_client.create_container(container)
        print(f"Created container: {container}")
    except Exception:
        print(f"Container already exists: {container}")

files_to_upload = [
    ("../data/raw/employees.csv", "raw-data"),
    ("../data/raw/payroll_records.csv", "raw-data"),
    ("../data/raw/manager_approvals.csv", "raw-data"),
    ("../data/raw/payroll_tickets.csv", "raw-data"),
    ("../data/processed/payroll_enriched.csv", "processed-data"),
    ("../data/documents/payroll_policy.txt", "documents"),
    ("../data/documents/data_dictionary.txt", "documents"),
]

for file_path, container_name in files_to_upload:
    blob_name = os.path.basename(file_path)
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    with open(file_path, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)

    print(f"Uploaded {blob_name} to {container_name}")

Container already exists: raw-data
Container already exists: documents
Container already exists: processed-data
Uploaded employees.csv to raw-data
Uploaded payroll_records.csv to raw-data
Uploaded manager_approvals.csv to raw-data
Uploaded payroll_tickets.csv to raw-data
Uploaded payroll_enriched.csv to processed-data
Uploaded payroll_policy.txt to documents
Uploaded data_dictionary.txt to documents


##### **Ingest structured data into Azure SQL**

In [3]:
# Ingest structured data into Azure SQL
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus

load_dotenv(override=True)

server = os.getenv("AZURE_SQL_SERVER")
database = os.getenv("AZURE_SQL_DATABASE")
username = os.getenv("AZURE_SQL_USERNAME")
password = os.getenv("AZURE_SQL_PASSWORD")

connection_string = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=no;"
    f"Connection Timeout=60;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=%s" % quote_plus(connection_string)
)


# Raw tables
employees_df = pd.read_csv("../data/raw/employees.csv")
approvals_df = pd.read_csv("../data/raw/manager_approvals.csv")
tickets_df = pd.read_csv("../data/raw/payroll_tickets.csv")

# Processed table (important one)
payroll_df = pd.read_csv("../data/processed/payroll_enriched.csv")

# Employees
employees_df["hire_date"] = pd.to_datetime(employees_df["hire_date"], errors="coerce")

# Payroll
payroll_df["pay_period_end"] = pd.to_datetime(payroll_df["pay_period_end"], errors="coerce")

# Approvals
approvals_df["submitted_date"] = pd.to_datetime(approvals_df["submitted_date"], errors="coerce")

# Tickets
tickets_df["created_date"] = pd.to_datetime(tickets_df["created_date"], errors="coerce")

# pushing it to SQL
employees_df.to_sql("employees", engine, if_exists="replace", index=False)
payroll_df.to_sql("payroll_enriched", engine, if_exists="replace", index=False)
approvals_df.to_sql("manager_approvals", engine, if_exists="replace", index=False)
tickets_df.to_sql("payroll_tickets", engine, if_exists="replace", index=False)

print("Data uploaded to Azure SQL successfully.")

Data uploaded to Azure SQL successfully.


In [4]:
# quick validation sql upload
query = """
SELECT TOP 3
    p.payroll_id,
    p.employee_id,
    e.employee_name,
    e.department,
    p.net_pay,
    p.risk_level,
    p.issue_summary
FROM payroll_enriched p
JOIN employees e
    ON p.employee_id = e.employee_id
ORDER BY p.risk_score DESC;
"""

pd.read_sql(query, engine)

,payroll_id,employee_id,employee_name,department,net_pay,risk_level,issue_summary
0,P002506,E0209,Tracy Anderson,Finance,1528.91,High,"Unusual Overtime, Missing timesheet, Pending m..."
1,P002560,E0214,Christian Pearson,Operations,1776.25,High,"Unusual Overtime, Pending manager approval, La..."
2,P002654,E0222,Nathan Reynolds,Finance,2735.32,High,"Negative Deduction, Missing timesheet, Large c..."


##### **Azure AI Vector Store ingestion For RAG**

In [5]:
# doc chunks
import os
from dotenv import load_dotenv

load_dotenv(override=True)

docs_folder = "../data/documents"

def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    
    return chunks

documents = []

for filename in os.listdir(docs_folder):
    if filename.endswith(".txt"):
        path = os.path.join(docs_folder, filename)
        
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        
        chunks = chunk_text(text)
        
        for i, chunk in enumerate(chunks):
            documents.append({
                "id": f"{filename.replace('.txt','')}_{i}",
                "source_file": filename,
                "content": chunk
            })

len(documents), documents[0]

(18,
 {'id': 'attendance_timesheet_policy_0',
  'source_file': 'attendance_timesheet_policy.txt',
  'content': 'ATTENDANCE AND TIMESHEET POLICY\n\nPurpose:\nThis policy defines timesheet submission expectations and payroll impact.\n\nTimesheet Submission:\nHourly employees must submit accurate timesheets before payroll cutoff. Missing or late timesheets may delay payroll processing.\n\nCorrections:\nTimesheet corrections require manager review. Corrections submitted after cutoff may be processed in a future pay cycle.\n\nPayroll Impact:\nIncorrect hours, missing punches, unapproved overtime, and late submissions may cause pay variance, delayed overtime, or payroll tickets.\n\nEmployee Responsibility:\nEmployees are responsible for reviewing hours before submission and promptly reporting discrepancies.'})

In [6]:
# Create embeddings using OpenAI
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

for doc in documents:
    doc["content_vector"] = get_embedding(doc["content"])

print("Embeddings created:", len(documents))
print("Vector length:", len(documents[0]["content_vector"]))

Embeddings created: 18
Vector length: 1536


In [7]:
#Create Azure AI Search vector index

from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile
)

search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_ADMIN_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")

index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=AzureKeyCredential(search_key)
)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="source_file", type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="payroll-vector-profile"
    )
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(name="payroll-hnsw")
    ],
    profiles=[
        VectorSearchProfile(
            name="payroll-vector-profile",
            algorithm_configuration_name="payroll-hnsw"
        )
    ]
)

index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search
)

try:
    index_client.delete_index(index_name)
    print("Existing index deleted.")
except Exception:
    print("No existing index found.")

index_client.create_index(index)
print("Azure AI Search index created.")

Existing index deleted.
Azure AI Search index created.


In [8]:
# Upload documents to Azure AI Search

from azure.search.documents import SearchClient

search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key)
)

result = search_client.upload_documents(documents=documents)

print("Documents uploaded:", len(result))

Documents uploaded: 18


In [9]:
# Test keyword search

results = search_client.search(
    search_text="overtime approval policy",
    top=3
)

for result in results:
    print("SOURCE:", result["source_file"])
    print("CONTENT:", result["content"][:500])
    print("-" * 80)

SOURCE: overtime_policy.txt
CONTENT: OVERTIME POLICY

Purpose:
This policy defines overtime eligibility, approval requirements, and payroll treatment.

Eligibility:
Non-exempt full-time and part-time employees are eligible for overtime when hours worked exceed applicable thresholds. Standard overtime is typically calculated at 1.5 times the regular hourly rate.

Approval Requirement:
Overtime should be approved by the employee's manager before payroll processing. Overtime without approval may be flagged for payroll review and may r
--------------------------------------------------------------------------------
SOURCE: manager_approval_guidelines.txt
CONTENT: MANAGER APPROVAL GUIDELINES

Purpose:
This document explains manager responsibilities for approving payroll-related exceptions.

Approval Areas:
Managers may need to approve overtime, corrected hours, missed punches, PTO requests, shift adjustments, and payroll exceptions.

Approval Timing:
Approvals should be completed before pay

In [10]:
# Test vector search

from azure.search.documents.models import VectorizedQuery

query = "When should overtime be flagged for payroll review?"
query_vector = get_embedding(query)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=3,
    fields="content_vector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["source_file", "content"],
    top=3
)

for result in results:
    print("SOURCE:", result["source_file"])
    print("CONTENT:", result["content"][:500])
    print("-" * 80)

SOURCE: overtime_policy.txt
CONTENT: OVERTIME POLICY

Purpose:
This policy defines overtime eligibility, approval requirements, and payroll treatment.

Eligibility:
Non-exempt full-time and part-time employees are eligible for overtime when hours worked exceed applicable thresholds. Standard overtime is typically calculated at 1.5 times the regular hourly rate.

Approval Requirement:
Overtime should be approved by the employee's manager before payroll processing. Overtime without approval may be flagged for payroll review and may r
--------------------------------------------------------------------------------
SOURCE: overtime_policy.txt
CONTENT: l cutoff
- Schedule mismatch
- Exception triggered by unusually high hours

Payroll Impact:
Approved overtime is included in the regular pay cycle. Pending or rejected overtime may delay or reduce overtime payment until reviewed.

Escalation:
Employees should contact their manager first for approval-related questions and payroll operations fo

## Enhancement v2 — Clean Azure Ingestion Pipeline

This section adds an idempotent ingestion flow for Blob upload, Azure SQL table loading, Azure AI Search vector indexing, and smoke tests. It assumes your containers already exist: `raw-data`, `documents`, and `processed-data`.


In [ ]:

# Enhancement v2: Clean, reusable Azure ingestion pipeline
# Run this section after generating CSVs and policy docs locally.

import os
import json
import time
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from azure.storage.blob import BlobServiceClient, ContentSettings
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
)
from azure.search.documents.models import VectorizedQuery
from openai import OpenAI

load_dotenv(override=True)

# -----------------------------
# Local folders
# -----------------------------
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
DOCS_DIR = Path("../data/documents")

# -----------------------------
# Azure config
# -----------------------------
AZURE_STORAGE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
AZURE_SQL_SERVER = os.getenv("AZURE_SQL_SERVER")
AZURE_SQL_DATABASE = os.getenv("AZURE_SQL_DATABASE")
AZURE_SQL_USERNAME = os.getenv("AZURE_SQL_USERNAME")
AZURE_SQL_PASSWORD = os.getenv("AZURE_SQL_PASSWORD")
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_ADMIN_KEY = os.getenv("AZURE_SEARCH_ADMIN_KEY")
AZURE_SEARCH_INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME", "payroll-policy-index")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

required = {
    "AZURE_STORAGE_CONNECTION_STRING": AZURE_STORAGE_CONNECTION_STRING,
    "AZURE_SQL_SERVER": AZURE_SQL_SERVER,
    "AZURE_SQL_DATABASE": AZURE_SQL_DATABASE,
    "AZURE_SQL_USERNAME": AZURE_SQL_USERNAME,
    "AZURE_SQL_PASSWORD": AZURE_SQL_PASSWORD,
    "AZURE_SEARCH_ENDPOINT": AZURE_SEARCH_ENDPOINT,
    "AZURE_SEARCH_ADMIN_KEY": AZURE_SEARCH_ADMIN_KEY,
    "OPENAI_API_KEY": OPENAI_API_KEY,
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"Missing environment variables: {missing}")

print("Config loaded successfully")
print("Search index:", AZURE_SEARCH_INDEX_NAME)


In [ ]:

# Upload CSVs and policy documents to Azure Blob Storage
blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)

containers = ["raw-data", "documents", "processed-data"]
for container in containers:
    try:
        blob_service_client.create_container(container)
        print(f"Created container: {container}")
    except Exception:
        print(f"Container exists or cannot be created: {container}")


def upload_folder_to_container(folder_path: Path, container_name: str, extensions: tuple):
    container_client = blob_service_client.get_container_client(container_name)
    uploaded = []

    for file_path in folder_path.glob("*"):
        if file_path.is_file() and file_path.suffix.lower() in extensions:
            content_type = "text/csv" if file_path.suffix.lower() == ".csv" else "text/plain"
            with open(file_path, "rb") as f:
                container_client.upload_blob(
                    name=file_path.name,
                    data=f,
                    overwrite=True,
                    content_settings=ContentSettings(content_type=content_type),
                )
            uploaded.append(file_path.name)

    return uploaded

raw_uploaded = upload_folder_to_container(RAW_DIR, "raw-data", (".csv",))
processed_uploaded = upload_folder_to_container(PROCESSED_DIR, "processed-data", (".csv",))
docs_uploaded = upload_folder_to_container(DOCS_DIR, "documents", (".txt",))

print("Raw uploaded:", raw_uploaded)
print("Processed uploaded:", processed_uploaded)
print("Documents uploaded:", docs_uploaded)


In [ ]:

# Load structured CSVs into Azure SQL
connection_string = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={AZURE_SQL_SERVER};"
    f"DATABASE={AZURE_SQL_DATABASE};"
    f"UID={AZURE_SQL_USERNAME};"
    f"PWD={AZURE_SQL_PASSWORD};"
    f"Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(connection_string),
    fast_executemany=True,
)

table_files = {
    "employees": RAW_DIR / "employees.csv",
    "payroll_records": RAW_DIR / "payroll_records.csv",
    "manager_approvals": RAW_DIR / "manager_approvals.csv",
    "payroll_tickets": RAW_DIR / "payroll_tickets.csv",
    "payroll_enriched": PROCESSED_DIR / "payroll_enriched.csv",
}

for table_name, file_path in table_files.items():
    if not file_path.exists():
        print(f"Skipping missing file: {file_path}")
        continue

    df = pd.read_csv(file_path)
    df.to_sql(table_name, con=engine, if_exists="replace", index=False, chunksize=500)
    print(f"Loaded {table_name}: {df.shape[0]} rows, {df.shape[1]} columns")

# Quick validation
with engine.connect() as conn:
    for table_name in table_files:
        try:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
            print(f"{table_name}: {count} rows")
        except Exception as e:
            print(f"Could not validate {table_name}: {e}")


In [ ]:

# Create searchable chunks from local policy documents

def chunk_text(text: str, chunk_size: int = 900, overlap: int = 150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

search_documents = []

for file_path in DOCS_DIR.glob("*.txt"):
    text_content = file_path.read_text(encoding="utf-8")
    chunks = chunk_text(text_content)

    for i, chunk in enumerate(chunks):
        search_documents.append({
            "id": f"{file_path.stem}-{i}".replace("_", "-").replace(".", "-"),
            "source_file": file_path.name,
            "doc_type": file_path.stem,
            "chunk_id": i,
            "content": chunk,
        })

print("Total chunks prepared:", len(search_documents))
print("Sample:", json.dumps(search_documents[0], indent=2)[:700])


In [ ]:

# Create embeddings and Azure AI Search vector index
openai_client = OpenAI(api_key=OPENAI_API_KEY)


def get_embedding(text: str):
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text,
    )
    return response.data[0].embedding

for doc in search_documents:
    doc["content_vector"] = get_embedding(doc["content"])

embedding_dim = len(search_documents[0]["content_vector"])
print("Embeddings created:", len(search_documents))
print("Embedding dimension:", embedding_dim)

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    credential=AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY),
)

# Delete old index if exists, then recreate cleanly
try:
    index_client.delete_index(AZURE_SEARCH_INDEX_NAME)
    print("Deleted existing index:", AZURE_SEARCH_INDEX_NAME)
    time.sleep(2)
except Exception:
    print("No existing index deleted or index did not exist")

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="source_file", type=SearchFieldDataType.String, filterable=True, sortable=True),
    SearchableField(name="doc_type", type=SearchFieldDataType.String, filterable=True, sortable=True),
    SimpleField(name="chunk_id", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=embedding_dim,
        vector_search_profile_name="vector-profile",
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],
    profiles=[VectorSearchProfile(name="vector-profile", algorithm_configuration_name="hnsw-config")],
)

index = SearchIndex(
    name=AZURE_SEARCH_INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
)

index_client.create_index(index)
print("Created Azure AI Search index:", AZURE_SEARCH_INDEX_NAME)

search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX_NAME,
    credential=AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY),
)

upload_result = search_client.upload_documents(search_documents)
print("Uploaded chunks:", len(upload_result))


In [ ]:

# Smoke tests: keyword, vector, and SQL queries

print("\n--- Keyword Search Test ---")
results = search_client.search(
    search_text="overtime approval payroll review",
    top=3,
    select=["source_file", "doc_type", "content"],
)
for r in results:
    print("SOURCE:", r["source_file"])
    print(r["content"][:400])
    print("-" * 80)

print("\n--- Vector Search Test ---")
query = "Why was my overtime reduced and what approval is required?"
query_vector = get_embedding(query)
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=3,
    fields="content_vector",
)
results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["source_file", "doc_type", "content"],
    top=3,
)
for r in results:
    print("SOURCE:", r["source_file"])
    print(r["content"][:400])
    print("-" * 80)

print("\n--- SQL Join Test ---")
sql_query = """
SELECT TOP 5
    p.payroll_id,
    p.employee_id,
    e.employee_name,
    e.department,
    p.overtime_hours,
    p.net_pay,
    p.risk_level,
    p.issue_summary
FROM payroll_enriched p
LEFT JOIN employees e
    ON p.employee_id = e.employee_id
ORDER BY p.risk_score DESC;
"""

display(pd.read_sql(sql_query, engine))
